# Tutorial 3: Tree-Based Methods for Market Timing

**Big Data in Finance** | Dr Daniele Bianchi  
**Duration:** 1 hour

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Fit Decision Trees, Random Forests, and Gradient Boosting for return prediction
2. Understand key hyperparameters and their effects on overfitting
3. Compare tree-based methods to Ridge regression (from Week 2)
4. Evaluate models using both **statistical** ($R^2_{OS}$) and **economic** (Sharpe ratio) metrics

> 💡 **Note (scope):** This tutorial uses a deliberately simplified setup for a one-hour session — a single chronological train/test split and a reduced set of predictors — so its numbers will not exactly reproduce the full walk-forward results in the lecture slides and notes.

---

## Part 1: Setup and Data Preparation

Same data and split as Week 2 — predicting next month's market return.

In [ ]:
# =============================================================================
# Import libraries
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tree-based models from scikit-learn
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Model selection and preprocessing
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Set plot style and random seed for reproducibility
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Libraries loaded!")

In [ ]:
# =============================================================================
# Load and prepare the Goyal-Welch-Zafirov data
# =============================================================================

# Load data from CSV file
df = pd.read_csv('Data_GWZ.csv')

# Convert DATE column to datetime and set as index
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.set_index('DATE')

# Define our predictors (features) - these are macroeconomic variables
# d_p: dividend-price ratio, e_p: earnings-price ratio, etc.
feature_names = ['d_p', 'd_y', 'e_p', 'b_m', 'tbl', 'lty', 'tms', 'dfy', 'svar', 'infl']

# Create the prediction target: NEXT month's return
# shift(-1) moves returns up by one row, so row t contains return from t+1
df['target'] = df['MktRf'].shift(-1)

# Drop the last row (it has NaN target since there's no "next month")
df = df.dropna(subset=['target'])

# Separate features (X) and target (y)
X = df[feature_names]
y = df['target']

# =============================================================================
# Create chronological train/test split (80% train, 20% test)
# IMPORTANT: We split by time, not randomly, to avoid look-ahead bias
# =============================================================================

split_point = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]

print(f"Training: {X_train.index[0].strftime('%Y-%m')} to {X_train.index[-1].strftime('%Y-%m')} ({len(X_train)} obs)")
print(f"Test:     {X_test.index[0].strftime('%Y-%m')} to {X_test.index[-1].strftime('%Y-%m')} ({len(X_test)} obs)")

In [ ]:
# =============================================================================
# Define the out-of-sample R² function (same as Week 2)
# =============================================================================

def oos_r2(y_true, y_pred, y_train_mean):
    """
    Calculate out-of-sample R² relative to the historical mean benchmark.
    
    R²_OOS = 1 - (Sum of Squared Errors) / (Total Sum of Squares)
    
    - Positive R²: Model beats the historical mean
    - Negative R²: Model is WORSE than just predicting the mean
    - Zero R²: Model equals the historical mean
    
    Parameters:
    -----------
    y_true : array-like, actual returns
    y_pred : array-like, predicted returns
    y_train_mean : float, historical mean from training data
    
    Returns:
    --------
    float : out-of-sample R²
    """
    ss_res = np.sum((y_true - y_pred)**2)          # Sum of squared residuals (model errors)
    ss_tot = np.sum((y_true - y_train_mean)**2)    # Total sum of squares (benchmark errors)
    return 1 - ss_res / ss_tot

# Calculate and store the historical mean (our benchmark)
train_mean = y_train.mean()
print(f"Historical mean return: {train_mean:.3f}%")

---

## Part 2: Ridge Regression Baseline

First, let's fit Ridge regression as our **linear baseline** from Week 2.

In [ ]:
# =============================================================================
# Fit Ridge Regression (linear baseline from Week 2)
# =============================================================================

# Ridge regression requires standardized features (mean=0, std=1)
# This ensures the penalty treats all features equally
scaler = StandardScaler()

# IMPORTANT: fit_transform on training data, then transform test data
# This prevents information leakage from test set
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit Ridge with alpha=1.0 (regularization strength)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

# Generate predictions on test set
pred_ridge = ridge.predict(X_test_scaled)

# Evaluate using out-of-sample R²
r2_ridge = oos_r2(y_test.values, pred_ridge, train_mean)
print(f"Ridge OOS R²: {r2_ridge*100:.2f}%")

---

## Part 3: Decision Trees

### 3.1 How Trees Work

Decision trees partition the feature space by asking yes/no questions:
- "Is dividend yield > 0.03?"
- "Is term spread < 0.01?"

Each **leaf** contains a prediction (mean of training observations in that region).

### 3.2 Key Hyperparameters

| Parameter | Effect | Typical Values |
|-----------|--------|----------------|
| `max_depth` | Limits tree depth | 2, 3, 4, 5 |
| `min_samples_leaf` | Min observations per leaf | 5, 10, 20 |

**Deeper trees = more complex = more overfitting risk**

In [ ]:
# =============================================================================
# Fit a Decision Tree with depth constraints (to prevent overfitting)
# =============================================================================

# NOTE: Unlike Ridge, trees do NOT require feature scaling!
# Trees only care about the ordering of values, not their magnitude

tree = DecisionTreeRegressor(
    max_depth=3,           # Maximum depth of 3 levels (limits complexity)
    min_samples_leaf=10,   # Each leaf must have at least 10 observations
    random_state=42        # For reproducibility
)

# Fit on training data (unscaled - trees don't need scaling)
tree.fit(X_train, y_train)

# Generate predictions
pred_tree = tree.predict(X_test)

# Evaluate
r2_tree = oos_r2(y_test.values, pred_tree, train_mean)
print(f"Decision Tree (depth=3) OOS R²: {r2_tree*100:.2f}%")

In [ ]:
# =============================================================================
# Visualize the tree structure
# =============================================================================

# This shows the decision rules the tree learned
# Each node shows: the split condition, MSE, number of samples, and predicted value

plt.figure(figsize=(14, 6))
plot_tree(
    tree,                        # The fitted tree model
    feature_names=feature_names, # Names to display instead of X[0], X[1], etc.
    filled=True,                 # Color nodes by predicted value
    rounded=True,                # Rounded corners on boxes
    fontsize=8                   # Font size for readability
)
plt.title('Decision Tree Structure (max_depth=3)')
plt.tight_layout()
plt.show()

### 3.3 The Overfitting Problem

What happens without depth constraints?

In [ ]:
# =============================================================================
# Demonstrate overfitting with an unconstrained tree
# =============================================================================

# Fit a tree with NO constraints - it will grow until each leaf is "pure"
tree_deep = DecisionTreeRegressor(random_state=42)  # No max_depth, no min_samples_leaf!
tree_deep.fit(X_train, y_train)

# Calculate IN-SAMPLE R² (how well it fits training data)
pred_train_deep = tree_deep.predict(X_train)
r2_in = 1 - np.sum((y_train - pred_train_deep)**2) / np.sum((y_train - y_train.mean())**2)

# Calculate OUT-OF-SAMPLE R² (how well it predicts new data)
pred_test_deep = tree_deep.predict(X_test)
r2_out = oos_r2(y_test.values, pred_test_deep, train_mean)

# Compare: huge gap = overfitting!
print(f"Deep Tree - In-sample R²:      {r2_in*100:.1f}%")
print(f"Deep Tree - Out-of-sample R²:  {r2_out*100:.1f}%")
print(f"\n⚠️ {r2_in*100:.0f}% in-sample but {r2_out*100:.0f}% out-of-sample = SEVERE OVERFITTING")
print(f"   The tree memorized the training data but learned nothing useful!")

---

## Part 4: Random Forest

### 4.1 The Idea

Random Forest reduces the variance of decision trees by:
1. **Bagging**: Train many trees on bootstrap samples
2. **Feature subsampling**: Each split considers only a random subset of features
3. **Averaging**: Final prediction = mean of all trees

### 4.2 Key Hyperparameters

| Parameter | Effect | Typical Values |
|-----------|--------|----------------|
| `n_estimators` | Number of trees | 100, 200, 500 |
| `max_depth` | Depth per tree | 3, 5, 7, None |
| `max_features` | Features per split | 'sqrt', 0.33, 0.5 |
| `min_samples_leaf` | Min obs per leaf | 5, 10, 20 |

**Key insight:** More trees never hurts — just slower. Focus tuning on `max_depth` and `max_features`.

In [ ]:
# =============================================================================
# Fit Random Forest
# =============================================================================

rf = RandomForestRegressor(
    n_estimators=200,       # Number of trees in the forest
                            # More trees = more stable predictions (but slower)
    
    max_depth=5,            # Maximum depth of each individual tree
                            # Shallower = less overfitting per tree
    
    max_features='sqrt',    # Number of features to consider at each split
                            # 'sqrt' means sqrt(10) ≈ 3 features per split
                            # This decorrelates trees (key to variance reduction!)
    
    min_samples_leaf=10,    # Minimum samples required in each leaf
                            # Higher = smoother predictions, less overfitting
    
    random_state=42,        # For reproducibility
    n_jobs=-1               # Use all CPU cores for parallel training
)

# Fit on training data (no scaling needed for trees!)
rf.fit(X_train, y_train)

# Generate predictions (each prediction = average of 200 tree predictions)
pred_rf = rf.predict(X_test)

# Evaluate
r2_rf = oos_r2(y_test.values, pred_rf, train_mean)
print(f"Random Forest OOS R²: {r2_rf*100:.2f}%")

### 🤖 AI Prompt Exercise 1: Understand Random Forest Hyperparameters

**Task:** Ask an AI to explain the role of `max_features` in Random Forest.

**Copy this prompt:**

```
In Random Forest, what does the max_features hyperparameter do?

Explain:
1. Why we don't use all features at each split
2. How this "decorrelates" the trees
3. What happens if max_features is too low vs too high

Keep it concise (4-5 sentences).
```

**Write the key points:**

*Your notes:*



---

## Part 5: Gradient Boosting

### 5.1 The Idea

Gradient Boosting builds trees **sequentially**:
1. Fit a tree to the data
2. Compute the residuals (errors)
3. Fit the next tree to the residuals
4. Repeat

Each tree corrects the mistakes of the previous ensemble.

### 5.2 Key Hyperparameters

| Parameter | Effect | Typical Values |
|-----------|--------|----------------|
| `n_estimators` | Number of boosting rounds | 50, 100, 200 |
| `max_depth` | Tree depth (keep shallow!) | 2, 3, 4 |
| `learning_rate` | Shrinkage factor | 0.01, 0.05, 0.1 |
| `subsample` | Row subsampling | 0.7, 0.8, 1.0 |

**Critical trade-off:** `learning_rate` × `n_estimators`
- Small learning rate + many trees = careful learning (usually better)
- Large learning rate + few trees = aggressive learning (faster but riskier)

In [ ]:
# =============================================================================
# Fit Gradient Boosting
# =============================================================================

gb = GradientBoostingRegressor(
    n_estimators=100,       # Number of boosting rounds (trees added sequentially)
                            # Unlike RF, more trees CAN lead to overfitting!
    
    max_depth=3,            # Depth of each tree - keep SHALLOW for boosting
                            # Boosting reduces bias, so we don't need deep trees
                            # Shallow trees = "weak learners"
    
    learning_rate=0.05,     # Shrinkage factor - how much each tree contributes
                            # 0.05 means each tree only contributes 5% to prediction
                            # Smaller = more regularization, need more trees
    
    subsample=0.8,          # Fraction of samples used per tree
                            # <1.0 adds randomness ("stochastic" gradient boosting)
                            # Helps prevent overfitting
    
    random_state=42
)

# Fit the model
gb.fit(X_train, y_train)

# Generate predictions
pred_gb = gb.predict(X_test)

# Evaluate
r2_gb = oos_r2(y_test.values, pred_gb, train_mean)
print(f"Gradient Boosting OOS R²: {r2_gb*100:.2f}%")

### 5.3 Overfitting in Gradient Boosting

Unlike Random Forest, Gradient Boosting **can overfit** with too many trees.

In [ ]:
# =============================================================================
# Demonstrate overfitting as we add more boosting rounds
# =============================================================================

# Test different numbers of trees
n_trees_list = [10, 25, 50, 100, 200, 500]
train_errors = []  # Store training errors
test_errors = []   # Store test errors

for n in n_trees_list:
    # Fit GB with n trees
    gb_temp = GradientBoostingRegressor(
        n_estimators=n,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )
    gb_temp.fit(X_train, y_train)
    
    # Calculate MSE on training and test sets
    train_mse = np.mean((y_train - gb_temp.predict(X_train))**2)
    test_mse = np.mean((y_test - gb_temp.predict(X_test))**2)
    
    train_errors.append(train_mse)
    test_errors.append(test_mse)

# Plot the results
plt.figure(figsize=(8, 4))
plt.plot(n_trees_list, train_errors, 'o-', label='Training Error', linewidth=2)
plt.plot(n_trees_list, test_errors, 's-', label='Test Error', linewidth=2)
plt.xlabel('Number of Trees')
plt.ylabel('Mean Squared Error')
plt.title('Gradient Boosting: More Trees Can Overfit!')
plt.legend()
plt.tight_layout()
plt.show()

print("💡 Key insight: Training error keeps decreasing, but test error may increase")
print("   This is why we need early stopping or cross-validation!")

---

## Part 6: Statistical Performance Comparison

In [ ]:
# =============================================================================
# Compare all models using out-of-sample R²
# =============================================================================

# Collect results in a dictionary
stat_results = {
    'Historical Mean': 0.0,              # Benchmark: R² = 0 by definition
    'Ridge': r2_ridge,                   # Linear model from Week 2
    'Decision Tree (depth=3)': r2_tree,  # Single constrained tree
    'Random Forest': r2_rf,              # Ensemble of decorrelated trees
    'Gradient Boosting': r2_gb           # Sequential boosting
}

# Print sorted by R² (best first)
print("Out-of-Sample R² Comparison:")
print("="*50)
for model, r2 in sorted(stat_results.items(), key=lambda x: x[1], reverse=True):
    # Positive R² = beats historical mean, Negative = worse than mean
    print(f"{model:25s}: {r2*100:+.2f}%")

---

## Part 7: Economic Performance

Statistical accuracy isn't everything! Let's evaluate **economic value** using a market timing strategy.

### 7.1 Market Timing Strategy

Mean-variance optimal weight: $\omega_t = \frac{1}{\gamma} \cdot \frac{\hat{r}_{t+1}}{\hat{\sigma}^2}$

With constraints: $-1 \leq \omega_t \leq 1.5$

In [ ]:
# =============================================================================
# Define function to compute market timing strategy performance
# =============================================================================

def compute_strategy_performance(predictions, actual, gamma=5, tc=0.005):
    """
    Compute performance metrics for a market timing strategy.
    
    The strategy uses mean-variance optimal weights:
    weight = (1/gamma) * (predicted_return / variance)
    
    Parameters:
    -----------
    predictions : array, predicted returns for each period
    actual : array, realized returns for each period
    gamma : float, risk aversion coefficient (higher = more conservative)
    tc : float, transaction cost per unit of weight change (e.g., 0.005 = 50 bps)
    
    Returns:
    --------
    dict : performance metrics (return, volatility, Sharpe, etc.)
    """
    # Use historical variance as our estimate
    variance = y_train.var()
    
    # Calculate optimal weights, then constrain to [-1, 1.5]
    # Positive weight = long stocks, Negative = short stocks
    raw_weights = (1/gamma) * (predictions / variance)
    weights = np.clip(raw_weights, -1, 1.5)  # Apply position limits
    
    # Portfolio returns = weight × market return
    port_ret = weights * actual
    
    # Calculate turnover (how much we trade)
    # Turnover = average absolute change in weights per month, annualized
    weight_changes = np.abs(np.diff(weights))
    turnover = np.mean(weight_changes) * 12  # Annualized
    
    # Calculate transaction costs
    # Cost = |weight change| × transaction cost rate
    costs = np.abs(np.diff(weights, prepend=weights[0])) * tc
    port_ret_net = port_ret - costs  # Returns after costs
    
    # Annualized performance metrics
    ann_ret = np.mean(port_ret) * 12 * 100        # Annualized return in %
    ann_vol = np.std(port_ret) * np.sqrt(12) * 100  # Annualized volatility in %
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0  # Sharpe ratio (gross)
    
    # Sharpe ratio after transaction costs
    ann_ret_net = np.mean(port_ret_net) * 12 * 100
    sharpe_net = ann_ret_net / ann_vol if ann_vol > 0 else 0
    
    return {
        'Ann. Ret (%)': ann_ret,
        'Ann. Vol (%)': ann_vol,
        'Sharpe': sharpe,
        'Sharpe (net)': sharpe_net,
        'Turnover': turnover
    }

In [ ]:
# =============================================================================
# Compute economic performance for all strategies
# =============================================================================

actual = y_test.values

# Historical mean strategy: always predict the same value
pred_mean = np.full(len(actual), train_mean)

# Calculate performance for each model
econ_results = []

for name, pred, r2 in [
    ('Historical Mean', pred_mean, 0.0),
    ('Ridge', pred_ridge, r2_ridge),
    ('Decision Tree', pred_tree, r2_tree),
    ('Random Forest', pred_rf, r2_rf),
    ('Gradient Boosting', pred_gb, r2_gb)
]:
    perf = compute_strategy_performance(pred, actual)
    perf['Model'] = name
    perf['R² OS (%)'] = r2 * 100
    econ_results.append(perf)

# Add Buy & Hold benchmark (always fully invested, weight=1)
bh_ret = np.mean(actual) * 12 * 100
bh_vol = np.std(actual) * np.sqrt(12) * 100
econ_results.append({
    'Model': 'Buy & Hold',
    'R² OS (%)': np.nan,  # Not applicable
    'Ann. Ret (%)': bh_ret,
    'Ann. Vol (%)': bh_vol,
    'Sharpe': bh_ret / bh_vol,
    'Sharpe (net)': bh_ret / bh_vol,  # No trading costs for B&H
    'Turnover': 0.0
})

# Create and display results table
perf_df = pd.DataFrame(econ_results)[['Model', 'R² OS (%)', 'Ann. Ret (%)', 'Ann. Vol (%)', 
                                       'Sharpe', 'Sharpe (net)', 'Turnover']]

print("Statistical vs Economic Performance:")
print("="*85)
print(perf_df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

### 🤖 AI Prompt Exercise 2: The R² vs Sharpe Puzzle

**Task:** Ask an AI to explain why better R² doesn't always mean better returns.

**Copy this prompt (fill in your values):**

```
I'm comparing models for market timing. Here's a puzzle:

- Random Forest has R² = [YOUR VALUE]%
- Ridge has R² = [YOUR VALUE]%
- But Ridge has higher Sharpe ratio than Random Forest

How can a model with worse statistical accuracy (R²) have better economic performance (Sharpe)?

Hints to consider:
- R² measures average squared error
- Random Forest averages many trees → predictions closer to zero
- Portfolio weights depend on prediction magnitude

Explain in 4-5 sentences.
```

**Write the key insights:**

*Your notes:*



---

## Part 8: Hyperparameter Tuning

### 🤖 AI Prompt Exercise 3: Tune Random Forest

**Task:** Ask an AI to write GridSearchCV code for Random Forest.

**Copy this prompt:**

```
Write Python code to tune Random Forest hyperparameters using GridSearchCV with time-series cross-validation.

I have:
- X_train, y_train (training data)
- oos_r2(y_true, y_pred, train_mean) function
- train_mean already defined

Search over:
- n_estimators: [100, 200]
- max_depth: [3, 5, 7]
- min_samples_leaf: [5, 10]

Use TimeSeriesSplit with 5 folds and neg_mean_squared_error scoring.
Print best parameters, then evaluate on X_test, y_test using oos_r2.
```

**Paste and run:**

In [ ]:
# Paste AI-generated code here:


---

## Part 9: Key Takeaways

### Summary

1. **Single Decision Trees** overfit severely without constraints
   - Use `max_depth` and `min_samples_leaf` to regularize

2. **Random Forest** reduces variance through bagging + feature subsampling
   - More trees is always fine (just slower)
   - Robust to hyperparameter choices

3. **Gradient Boosting** reduces bias by fitting residuals
   - *Can overfit* with too many trees
   - Use small learning rate + early stopping

4. **Statistical accuracy ≠ Economic value**
   - RF may have best R² but Ridge has best Sharpe
   - Always evaluate both!

### When Do Trees Excel?

| Setting | Trees vs Linear |
|---------|----------------|
| Market timing (low signal) | Competitive but not dominant |
| Cross-sectional (large N) | Often beat linear models |
| Credit risk (non-linear thresholds) | Can capture threshold effects |

---

## Next Week

**Topic:** Classification Methods for Credit Risk

**Key difference:** Credit risk has *higher signal-to-noise* — trees may help more!

**Reading:** James et al. (2023), Chapter 4